# 01. ChromaGAN 파이프라인

**Paper**: ChromaGAN: Adversarial Picture Colorization with Semantic Class Distribution

원본 코드: https://github.com/pvitoria/ChromaGAN (TF2)  
본 프로젝트가 사용한 코드: https://github.com/superhighlevel/ChromaGan_Pytorch_Remake (PyTorch 재구현)

## 학습 결과 요약 (보고서 기준)
| 항목 | 값 |
| --- | --- |
| epochs | 10 |
| batch size | 1 (OOM 회피) |
| learning rate | 2e-5 |
| train 수량 | 12,922 |
| test 수량 | 2,465 |
| 환경 | Colab Pro |

Discriminator loss 는 후반부로 갈수록 발산 경향, Generator loss 는 -0.05 부근에서 안정. 정성 평가에서는 fine-tune 모델이 기본 weight 보다 한국 음식 텍스처를 더 잘 살림.

## 1. 환경 셋업

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/딥러닝
!git clone https://github.com/superhighlevel/ChromaGan_Pytorch_Remake.git ChromaGAN_run || true
%cd ChromaGAN_run
!pip install -q torch torchvision tqdm pillow

## 2. config.py 수정

팀이 사용한 하이퍼파라미터로 덮어씁니다.

In [ ]:
config_text = '''
import torch
GRADIENT_PENALTY_WEIGHT = 10
EPOCHS = 10
MODEL_DIR = 'models'
MODEL_C = MODEL_DIR + '/colorization_10.pt'
MODEL_D = MODEL_DIR + '/discriminator_10.pt'
TRAIN_PATH = '../DATASET/train/color_image'
TEST_PATH  = '../DATASET/test/color_image'
TR_PATH = 'train/'
TE_PATH = 'test/'
OUTPUT_PATH = 'Output_10/' + TR_PATH
BATCH_SIZE = 1
TEST_BATCH_SIZE = 1
CHECK_PER = 100
LR = 2e-5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
'''
open('config.py', 'w').write(config_text)

## 3. 학습

Generator 와 Discriminator loss 를 print 로 모니터링합니다 (pickle 저장은 OOM 으로 비활성).

In [ ]:
!python train.py 2>&1 | tee train.log

## 4. 추론 (한국 음식 test set)

In [ ]:
!python test.py --weight models/colorization_10.pt --input ../DATASET/test/gray_image --output Output_10/test

## 5. 결과 비교 시각화

기존 weight vs 10 epoch fine-tune vs 원본 컬러

In [ ]:
import matplotlib.pyplot as plt, glob, os
from PIL import Image
test_gray = sorted(glob.glob('../DATASET/test/gray_image/*.png'))[:6]
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i, gp in enumerate(test_gray):
    base = os.path.basename(gp).replace('_gray.png', '')
    color_orig = f'../DATASET/test/color_image/{base}_color.png'
    pred = f'Output_10/test/{base}.png'
    axes[0, i].imshow(Image.open(gp), cmap='gray'); axes[0, i].axis('off')
    if os.path.exists(pred):
        axes[1, i].imshow(Image.open(pred))
    axes[1, i].axis('off')
    axes[2, i].imshow(Image.open(color_orig)); axes[2, i].axis('off')
axes[0, 0].set_ylabel('gray', fontsize=12)
axes[1, 0].set_ylabel('10 epoch fine-tune', fontsize=12)
axes[2, 0].set_ylabel('original color', fontsize=12)
plt.tight_layout(); plt.show()

## 메모

* Loss 값을 pickle 로 저장하려 했으나 OOM 으로 저장 실패 → print 로 확인.
* Discriminator loss 가 후반에 발산하지만 Generator 결과는 안정적이라 학습은 종료까지 진행했습니다.